Read API Structure
DataFrameReader.format(...).oprions("key", "value").schema(...),load()

Write API Structure
DataFrameWriter.format(...).options("key", "value").mode(...).partitionBy(...).bucketBy(...).sortBy(...).save()

In [29]:

from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark 

In [30]:
northwind_folder = "s3a://lakehouse/external-locations/sample-data/northwind/"

spark.sql("show catalogs").show()
spark.sql("show schemas").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

+------------+
|   namespace|
+------------+
|     default|
|northwind_db|
+------------+



In [31]:
spark.sql("create schema if not exists northwind_db")

spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")

DataFrame[]

In [ ]:
from pyspark.sql.functions import col, to_date, to_timestamp

spark.sql("use schema northwind_db")

df = (spark.read
      .format("csv")
      .option("header", "true")
      .load(f"{northwind_folder}Orders.csv")
      .withColumn("OrderDate", to_date(to_timestamp(col("OrderDate")), "format='yyyy-MM-dd'"))
      .withColumn("RequiredDate", to_date(to_timestamp(col("RequiredDate")), "format='yyyy-MM-dd'"))
      #.withColumn("ShippedDate", to_date(col("ShippedDate").cast("timestamp"), "format='yyyy-MM-dd'"))
      #.withColumn("ShipVia", col("ShipVia").cast("int"))
      .withColumn("Freight", col("Freight").cast("decimal(10,2)"))
      )

df.printSchema()

(df.write
 .format("delta")
 .option("overwriteSchema", "true")
 .mode("overwrite")
 .partitionBy("CustomerID")
 .saveAsTable("orders")
 )
spark.sql("show tables").show()



root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- RequiredDate: date (nullable = true)
 |-- ShippedDate: string (nullable = true)
 |-- ShipVia: string (nullable = true)
 |-- Freight: decimal(10,2) (nullable = true)
 |-- ShipName: string (nullable = true)
 |-- ShipAddress: string (nullable = true)
 |-- ShipCity: string (nullable = true)
 |-- ShipRegion: string (nullable = true)
 |-- ShipPostalCode: string (nullable = true)
 |-- ShipCountry: string (nullable = true)

+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|northwind_db|order_details|      false|
|northwind_db|       orders|      false|
+------------+-------------+-----------+



In [33]:


spark.sql("select * from orders").limit(3).show()

+-------+----------+----------+----------+------------+--------------------+-------+-------+--------------------+------------------+--------------+----------+--------------+-----------+
|OrderID|CustomerID|EmployeeID| OrderDate|RequiredDate|         ShippedDate|ShipVia|Freight|            ShipName|       ShipAddress|      ShipCity|ShipRegion|ShipPostalCode|ShipCountry|
+-------+----------+----------+----------+------------+--------------------+-------+-------+--------------------+------------------+--------------+----------+--------------+-----------+
|  10248|     VINET|         5|1996-07-04|  1996-08-01|1996-07-16 00:00:...|      3|  32.38|Vins et alcools C...|59 rue de l'Abbaye|         Reims|      NULL|         51100|     France|
|  10249|     TOMSP|         6|1996-07-05|  1996-08-16|1996-07-10 00:00:...|      1|  11.61|  Toms Spezialitäten|     Luisenstr. 48|       Münster|      NULL|         44087|    Germany|
|  10250|     HANAR|         4|1996-07-08|  1996-08-05|1996-07-12 00:0

In [ ]:

df = (spark.read.csv(f"{northwind_folder}OrdersDetails.csv", header=True)
 .withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)"))
 .withColumn("Quantity", col("Quantity").cast("int"))
 .withColumn("Discount", col("Discount").cast("decimal(10,2)"))
 )

df.printSchema()

(df.write
 .format("delta")
 .option("overwriteSchema", "true")
 .mode("overwrite")
 .partitionBy("ProductID")
 .saveAsTable("order_details"))

spark.sql("show tables").show()
spark.sql("select * from order_details").limit(3).show()

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: decimal(10,2) (nullable = true)

+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|northwind_db|order_details|      false|
|northwind_db|       orders|      false|
+------------+-------------+-----------+

+-------+---------+---------+--------+--------+
|OrderID|ProductID|UnitPrice|Quantity|Discount|
+-------+---------+---------+--------+--------+
|  10248|       11|    14.00|      12|    0.00|
|  10248|       42|     9.80|      10|    0.00|
|  10248|       72|    34.80|       5|    0.00|
+-------+---------+---------+--------+--------+

